# The product tour — from one sentence to RF waveforms

> *"I want a 10×10 array to lift 10 µm out of the focal plane, traverse 40 × 25 µm, and drop.
> Give me the waveforms."*

That is the whole brief, and this page is the whole product: **one call**, `plan_motion`, and
then what a lab does with what comes back. Notebooks 01–05 derive the physics; this page
uses it.

| § | What you get |
|---|--------------|
| 1 | the ask in eight lines, and the report that says what was built and what to watch |
| 2 | deliverable 1 — the parametric waveform file, and its expansion for the AWG |
| 3 | deliverable 2 — the simulation, and a small movie of the tweezers |
| 4 | where each piece of the physics is derived and checked |
| 5 | the fidelity limits, with this drive's own numbers |

Physics reference: arXiv:2510.11451 (equations `S#` refer to its Supplement; `Eq. 1` is in the
main text). The lab-facing manual is [`docs/guide.md`](../docs/guide.md).

In [ ]:
import json
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aodl import (
    ArraySpec,
    Lift,
    TrajectorySpec,
    Translate,
    WaveformSet,
    auto_grid,
    default_1030,
    plan_motion,
)
from aodl.units import MHz, um, us
from aodl.waveform.export import DEFAULT_SAMPLE_RATE, sample_times

P = default_1030()                  # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics = P.optics
tau = P.channels["Ax"].transit_time
OUT = Path("outputs")               # examples/outputs/ - gitignored
OUT.mkdir(exist_ok=True)


def with_order(params, order):
    "The same hardware with a different weak-drive expansion order (params.py)."
    return replace(params, channels={name: replace(a, mixing_order=order)
                                     for name, a in params.channels.items()})


# ---- the ask ------------------------------------------------------------------------------
story = TrajectorySpec(
    array=ArraySpec(10, 10, delta_f_x=1.0 * MHz, delta_f_y=1.3 * MHz),   # 10.3 x 13.4 um pitch
    moves=(Lift(10 * um, 150 * us),                       # up, out of the focal plane
           Translate(40 * um, 25 * um, 250 * us),         # across
           Lift(-10 * um, 150 * us)),                     # and back down
)
plan = plan_motion(story)                                 # <- the product, in one call
print(plan.report.summary())
# -------------------------------------------------------------------------------------------

Read that block from the top. **Mode `shepard`**: `plan_motion` tried the plain Eq. S19 drive
first and the band refused it — this move asks for $|\int Z\,dt| = 4.0\times10^{-9}$ m·s,
nearly twice the $2.06\times10^{-9}$ Eq. 1 allows (206 µs at 10 µm) — so it fell back to the
fading-Shepard ladders of Eqs. S24–S28 by itself, recording the refusal in
`plan.report.description`. **93 tones** on four channels, all of them inside the band with
1.3 MHz to spare on the tightest one. **34 hand-overs**, each with the shadow tweezers it
lights and the axis it lights them on. And five caveats that apply to *this* drive, not to the
package in general — the retardation lag, the extended grid, the even-$M$ comb offset, the
Table II rectangles, the shadows.

Everything below comes out of that one object.

## 2. Deliverable 1 — the waveform file

`plan.save` writes the **parametric function representation** (`docs/waveform_format.md`): per
channel, one row per polynomial segment of each tone's frequency law, one row per tone for its
phase and envelope, one row per segment of each fading rung's *fade coordinate* $g(t)$, and a
JSON snapshot of the hardware it was designed for. **No samples.** Expanding to an AWG buffer
is a separate step, and the two differ by ~60× in size here — a gap that grows with duration
and sample rate, because the parametric file does not grow at all.

In [ ]:
npz = plan.save(OUT / "06_product_tour.npz")
with np.load(npz) as f:
    meta = json.loads(str(f["meta"]))
    contents = {key: f[key].shape for key in f.files if key != "meta"}
print(f"{npz}  ({npz.stat().st_size / 1e3:.1f} kB, schema v{meta['schema_version']})")
for key, shape in contents.items():
    print(f"  {key:16s} {str(shape):10s}  " + ("(tone, t0, T, degree, c0..c9)" if "poly" in key
          or key.endswith("segments") else "(tone, phase0, env kind, env params)"))

back = WaveformSet.load(npz)                       # round-trip: float64-identical laws
probe = np.linspace(*plan.wfs.t_span, 401)
worst = max(float(np.max(np.abs(back.channels[n].eval_table(probe)["f"]
                                - cw.eval_table(probe)["f"])))
            for n, cw in plan.wfs.channels.items())
assert worst == 0.0

window = plan.render_samples(rate=DEFAULT_SAMPLE_RATE, t_span=(0.0, 2 * us))
n_full, _ = sample_times(plan.wfs.t_span, DEFAULT_SAMPLE_RATE)
full_bytes = n_full * len(plan.wfs.channels) * 4                       # float32 per channel
print(f"\nround trip           max |f_reloaded - f| = {worst:.1e} Hz")
print(f"AWG render           {DEFAULT_SAMPLE_RATE / 1e6:.0f} MS/s, "
      f"{len(window)} channels x {len(window['Ax']):,} samples for the 2 us window shown")
print(f"the whole drive      {n_full:,} samples per channel = {full_bytes / 1e6:.2f} MB "
      f"({full_bytes / npz.stat().st_size:.0f}x the parametric file)")
print(f"peak sample          {max(float(np.max(np.abs(v))) for v in window.values()):.3f} "
      f"(all four channels share one normalization, so their balance survives)")

And the picture that goes with it: every tone's frequency against time with **opacity following
its envelope**, so a fading ladder shows only the rungs that are actually driven — the analytic
spectrogram, with no FFT anywhere (`CLAUDE.md`) — over the band each channel is allowed, with
the live span drawn inside it. Watch the ladders walk upward together during the lift, hold
their slope through the traverse, and come back down; each rung dies at the top of the window
as its neighbour is born at the bottom.

In [ ]:
fig = plan.report.figure()
fig

## 3. Deliverable 2 — what the tweezers will do

`plan.simulate()` expands the drive into pupil terms at each frame's **retarded** time
$t_c = t - \tau/2$ and reduces them to one `SpotMetrics` per optical-frequency group. With no
argument it picks ~40 frames over $[\tau,\ T + \tau/2]$: starting at a full aperture transit
skips the fill transient, and ending half a transit late is what lets the *last requested
instant* be observed.

One modelling choice worth stating out loud. The product default is `mixing_order=3` — the
crystal compresses and makes IM3 ghosts — but a 93-tone drive expanded to third order is an
enormous term census, so the quick look below runs the same waveforms through the strictly
linear model (`mixing_order=1`: one tone, one beam), which is the right model for Eq. S19
*geometry*. Notebook 04 §5 shows exactly what order 3 adds to an array.

In [ ]:
quick = plan_motion(story, with_order(P, 1))          # same drive, cheaper crystal model
assert quick.report.tone_counts == plan.report.tone_counts

run = quick.simulate()                                # ~40 frames over [tau, T + tau/2]
t_ret = run.times - 0.5 * tau                         # what the drive was asked for
_, _, z_req = story.compile()

z_err = np.abs(run.tracked_z() - np.asarray(z_req(t_ret)))
astig = np.array([max(abs(m.delta_f) for m in frame) for frame in run.metrics])
groups = sorted({len(frame) for frame in run.metrics})

print(f"groups per frame        {groups}   (the 11 x 11 extended grid, see section 5)")
print(f"axial tracking error    {z_err.max() / optics.rayleigh:.1e} z_R "
      f"({z_err.max() / um:.1e} um)")
print(f"astigmatism |Delta F|   {astig.max() / optics.rayleigh:.1e} z_R   <- Table I says 0")
assert z_err.max() < 1e-3 * optics.rayleigh
assert astig.max() < 1e-3 * optics.rayleigh

In [ ]:
fig, (ax_z, ax_e) = plt.subplots(1, 2, figsize=(11.4, 3.8))

ax_z.plot(run.times / us, np.asarray(z_req(t_ret)) / um, color="k", lw=3, alpha=0.25,
          label=r"requested $Z(t-\tau/2)$")
ax_z.plot(run.times / us, run.tracked_z() / um, color="#3a7bd5", lw=1.4,
          label=r"simulated $\bar Z$")
for event in plan.report.fade_events:
    ax_z.axvline((event.time + 0.5 * tau) / us, color="#8ac926" if event.axis == "x" else
                 "#f4a261", lw=0.6, alpha=0.5)
ax_z.set(xlabel="t [µs]", ylabel="Z lab [µm]",
         title="150 / 250 / 150 µs at 10 µm (ticks: hand-overs, x / y)")
ax_z.legend(fontsize=8, loc="lower center")

ax_e.semilogy(run.times / us, np.maximum(z_err / optics.rayleigh, 1e-18), color="#3a7bd5",
              lw=1.4, label=r"$\bar Z$ error [$z_R$]")
ax_e.semilogy(run.times / us, np.maximum(astig / optics.rayleigh, 1e-18), color="#c1121f",
              lw=1.4, label=r"$|\Delta F|$ [$z_R$]")
ax_e.set(xlabel="t [µs]", ylabel="error", ylim=(1e-17, 1e-3),
         title="the geometry is exact to machine precision")
ax_e.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

### The movie

A small one — 28 frames on a coarse grid, about half a minute to render. The XY plane follows
the scene's own best focus, **hue carries each group's lab $Z$** (white in the plane, red
above), the XZ slice sits beside it and the four drives run underneath with a cursor on the
current frame. The first $\tau = 11.54$ µs is the fill transient: black until the
counter-propagating wavefronts meet at $\tau/2$, then a fast rise as the apertures fill.

Notebooks 04 and 05 render the full-resolution versions of this scene; `examples/outputs/` is
gitignored, so both live only where they were made.

In [ ]:
from IPython.display import Video

movie_run = quick.simulate(np.linspace(0.0, quick.wfs.t_span[1], 28))
grid = auto_grid(movie_run, long_side=200)
print(f"grid {grid.nx} x {grid.ny} px over {(grid.x1 - grid.x0) / um:.0f} x "
      f"{(grid.y1 - grid.y0) / um:.0f} um, {len(movie_run.metrics[-1])} groups per frame")

movie = quick.movie(OUT / "06_product_tour.mp4", times=movie_run.times, grid=grid,
                    mode="tracked", fps=12, xz_shape=(88, 60), spectrogram_panel=True, dpi=100)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB)")
Video(str(movie), embed=True, html_attributes="controls loop")

## 4. Where the physics lives

Nothing above is new physics — it is a composition of five notebooks' worth, each of which
verifies its own claims against closed form rather than against the code:

| Notebook | What it establishes | The check that matters |
|----------|---------------------|------------------------|
| [01 single AOD](01_single_aod_sweep.ipynb) | one deflector: deflection $\propto f(t-\tau/2)$, a chirp is a cylindrical lens, the aperture fills in a beam transit | fitted focal split vs $\Delta F = \lambda F^2\dot f/v^2$ |
| [02 crossed pair](02_crossed_pair_diagonal.ipynb) | two AODs: tone ladders are arrays, a diagonal chirp is a *spherical* defocus, IM3 ghosts at $f_j+f_k-f_i$ | Schroeder phases suppress the ghosts 57–437× vs random |
| [03 the 3D-AODL](03_aodl_3d_motion.ipynb) | four AODs, Table I: co-chirp → pure $Z$, counter-chirp → lateral motion with no focal shift | $\sigma_{\rm astig}$ at machine zero through arbitrary 3D moves |
| [04 the user story](04_array_lift_traverse.ipynb) | this same brief at Eq. 1 pace (25/30/25 µs), deliverables and the `mixing_order=3` census | tracking to $10^{-13}$ waists at 97 % band usage |
| [05 fading Shepard](05_fading_shepard.ipynb) | the ladders themselves: $\cos^p$ windows, $p_A+p_B=1$, shadow tweezers, interlacing, $\rho$ | a millisecond at 10 µm, total power flat to 0.43 % |

`docs/guide.md` is the same material as a manual; `docs/conventions.md` fixes every sign and
the retarded-time bookkeeping; `docs/waveform_format.md` specifies the file this page wrote.

## 5. Fidelity and limits, honestly

| Assumption | Where it bites | The number |
|------------|----------------|------------|
| weak drive: $e^{iCV}$ to `mixing_order` (default 3) | compression $\sim C^2/8$ per tone (≈1 % at $C = 0.3$) and IM3 ghosts (Eqs. S20–S22); flat efficiency across the band | notebook 02 §3; full coupled-mode Bragg is out of scope |
| quadratic pupil phase (coma dropped) | strongly curved chirps | vs the Eq. S11 quadrature reference: ~$10^{-15}$ relative on static tones |
| degree-2 envelope expansion across the aperture | **fast fades**, measured by $\rho = (w_{in}/v)/T_{\rm fade}$ | flatness $\propto \rho^2$; 1 % at $\rho \approx 0.057$ (notebook 05 §7) |
| the $\alpha_1$ tilt term at a clamped fade edge | the residual weight of a nearly-off rung | estimate $S^2(1+S^2)^{-p}(p\frac{\pi}{2}\rho)^2/4$ = 6.1e-4 vs 8.6e-4 measured — an estimate, ~30 % optimistic |
| Table II rectangles ($p_B = 0$) on an array ladder | rungs switch on and off instantaneously | ≈ **−40 dB** out-of-band splatter; `switch_ramp=<seconds>` smooths it, at the cost of a drive the v2 NPZ cannot store (`save` refuses it by name) |
| ideal geometry | overlaid apertures, matched delays ($x_{\rm err} = 0$, Eq. S29), scalar paraxial optics | — |

Two of those are properties of **this** drive rather than of the package, and the plan knows
both. For an array axis the Shepard ladder *is* the array ladder (Eq. S27), so $\Delta f$ is
fixed by the pitch you asked for — 1.0 and 1.3 MHz here, five to eight times narrower than the
free-axis ladders notebook 05 could choose, which makes these fades fast:

In [ ]:
fdot_z = 10 * um / (2 * P.lens_scale)                 # the co-chirp a 10 um hold costs (Eq. 1)
rho = {axis: (optics.w_in / P.sound_speed) * fdot_z / (0.5 * df)
       for axis, df in (("x", story.array.delta_f_x), ("y", story.array.delta_f_y))}

x_fades = [e.time for e in plan.report.fade_events
           if e.axis == "x" and story.duration / 3 < e.time < 2 * story.duration / 3]
probes = np.linspace(x_fades[0], x_fades[2], 41) + 0.5 * tau     # two x hand-overs, observed
across = quick.simulate(probes)
typical = np.array([float(np.median([m.power for m in frame])) for frame in across.metrics])
total = np.array([float(np.sum([m.power for m in frame])) for frame in across.metrics])

print(f"fade speed          rho = {rho['x']:.2f} (x), {rho['y']:.2f} (y)   "
      f"[1 % flatness sits at rho = 0.057]")
print(f"hand-over period    {(x_fades[1] - x_fades[0]) / us:.1f} us on x (= delta_f / fdot_Z); "
      f"{(probes[-1] - probes[0]) / us:.0f} us watched, {len(probes)} frames")
print(f"typical trap power  {100 * np.ptp(typical) / typical.mean():.1f} % ripple through them")
print(f"whole scene         {100 * np.ptp(total) / total.mean():.1f} % - the extended columns "
      f"trading, not the array flickering")

So: **the interior of the array holds its light to within about 8 % through every hand-over,
and the scene's total swings by a quarter** as the extended columns trade. Note that the first
number is *better* than the $\rho^2$ law would suggest — that law was fitted to the
single-tweezer windows of notebook 05 ($p_A = p_B = \tfrac12$), and extrapolating it from
$\rho = 0.057$ to $\rho = 0.30$ would predict some 28 %; an array's $(p_A, p_B) = (1, 0)$ pair
has no divergent shoulder to clamp and does better. Measure your own array rather than
extrapolating either way — notebook 05 §7 shows how, and it is four lines.

That is the honest shape of a Shepard array, and it is why the two operational rules matter:

* the grid you get is wider than the grid you asked for — $M+1$ columns at every instant for
  even $M$ (11 here), $M+2$ during a hand-over for odd $M$ — and the extra columns are real
  light, not artefacts;
* **never schedule a pick-up inside a fade zone.** Shadow tweezers sit at
  $\pm(\lambda F/v)\Delta f$ = ±10.3 µm (x) and ±13.4 µm (y) and reach *half* a trap's power
  mid-fade (Eq. S31). `plan.report.fade_events` is the timetable; the plateaus between events
  are where an experiment should act.

One more, specific to unequal spacings: Table II interlaces the two axes with $\xi_y = 1/2$,
which tiles exactly only when $\Delta f_x = \Delta f_y$. Here they differ (deliberately — equal
spacings would make every anti-diagonal of the array one coherent group), so the two schedules
beat and some hand-overs light both axes at once. `docs/guide.md` §6 and notebook 05 §5 show
what that costs.

---

**In one line:** a 10×10 array, 10 µm out of plane and 47 µm across, at a pace Eq. 1 cannot
buy — 93 tones inside a ±10 MHz band, tracking the request to $10^{-14}$ Rayleigh ranges with
$|\Delta F|$ at machine zero, in a 97 kB file and one movie.